# Reflection vs Reflexion Agents

## Overview

| **Aspect** | **Reflection** | **Reflexion** |

|------------|---------------|---------------|

| **Focus** | Improve current output | Learn from past attempts |

| **Memory** | Short-term (session) | Long-term (episodic) |

| **Pattern** | Generate → Critique → Refine | Attempt → Evaluate → Learn → Retry |

| **Feedback** | Self-critique | External evaluation |

| **Use Case** | Single-shot improvement | Multi-attempt learning |

---

## Visual Difference

In [ ]:
**Reflection:**
Task → Generate → Self-Critique → Refine → Output
        ↑__________________________|  (loop until good)

In [ ]:
**Reflexion:**
Task 1 → Attempt → Fail → Store learning
Task 2 → Retrieve past → Attempt → Success → Store
Task 3 → Use all learnings → Better attempt

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import os

load_dotenv()

api_key = os.environ['UNIFIED_LLM_KEY']
base_url = ""

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,
    max_tokens=500,
    api_key=api_key,
    base_url=base_url
)

---

# 1. Reflection Agent

**Pattern:** Generate → Self-Critique → Refine (current task only)

**Goal:** Improve the quality of a single output through self-reflection

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
from langchain_core.messages import HumanMessage, AIMessage

# State for Reflection Agent
class ReflectionState(TypedDict):
    task: str
    draft: str
    critique: str
    final_output: str
    iteration: int
    max_iterations: int

# Node 1: Generate initial draft
def generate_draft(state: ReflectionState):
    """Generate initial output"""
    print(f"\n📝 Iteration {state['iteration']}: Generating draft...")

    if state['iteration'] == 1:
        # First draft
        response = llm.invoke([
            HumanMessage(content=f"Write a short paragraph about: {state['task']}")
        ])
    else:
        # Refined draft based on critique
        response = llm.invoke([
            HumanMessage(content=f"""
            Task: {state['task']}

            Previous draft:
            {state['draft']}

            Critique:
            {state['critique']}

            Please improve the draft based on the critique.
            """)
        ])

    draft = response.content
    print(f"Draft: {draft}")
    return {"draft": draft}

# Node 2: Self-critique
def critique_draft(state: ReflectionState):
    """Agent critiques its own output"""
    print(f"\n🔍 Iteration {state['iteration']}: Self-critiquing...")

    response = llm.invoke([
        HumanMessage(content=f"""
        Review this draft and provide constructive criticism:

        {state['draft']}

        Focus on:
        - Clarity and coherence
        - Completeness
        - Grammar and style

        Provide specific suggestions for improvement.
        """)
    ])

    critique = response.content
    print(f"Critique: {critique}")
    return {"critique": critique}

# Node 3: Check if refinement needed
def should_continue(state: ReflectionState) -> Literal["refine", "finish"]:
    """Decide if we need another iteration"""
    if state['iteration'] >= state['max_iterations']:
        print(f"\n✅ Max iterations ({state['max_iterations']}) reached. Finishing...")
        return "finish"

    # Simple check: if critique is short, assume it's good
    if len(state['critique']) < 100:
        print("\n✅ Critique is minimal. Output is good!")
        return "finish"

    print(f"\n🔄 Refining based on critique...")
    return "refine"

# Node 4: Finalize output
def finalize_output(state: ReflectionState):
    """Mark final output"""
    print("\n✅ Finalizing output...")
    return {"final_output": state['draft']}

# Node 5: Increment iteration
def increment_iteration(state: ReflectionState):
    """Increment iteration counter"""
    return {"iteration": state['iteration'] + 1}

# Build Reflection Agent Graph
reflection_workflow = StateGraph(ReflectionState)

# Add nodes
reflection_workflow.add_node("generate", generate_draft)
reflection_workflow.add_node("critique", critique_draft)
reflection_workflow.add_node("increment", increment_iteration)
reflection_workflow.add_node("finalize", finalize_output)

# Add edges
reflection_workflow.add_edge(START, "generate")
reflection_workflow.add_edge("generate", "critique")
reflection_workflow.add_conditional_edges(
    "critique",
    should_continue,
    {
        "refine": "increment",
        "finish": "finalize"
    }
)
reflection_workflow.add_edge("increment", "generate")  # Loop back
reflection_workflow.add_edge("finalize", END)

# Compile
reflection_agent = reflection_workflow.compile()

In [ ]:
# Visualize the Reflection Agent graph
from IPython.display import Image, display

try:
    # Method 1: get_graph() returns ASCII representation
    display(Image(reflection_agent.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Visualization not available: {e}")
    print("\nGraph Structure (Text):")
    print(reflection_agent.get_graph().draw_ascii())

## Test Reflection Agent

In [ ]:
# Run Reflection Agent
print("=" * 60)
print("REFLECTION AGENT: Iterative Self-Improvement")
print("=" * 60)

result = reflection_agent.invoke({
    "task": "The benefits of artificial intelligence in healthcare",
    "draft": "",
    "critique": "",
    "final_output": "",
    "iteration": 1,
    "max_iterations": 3
})

print("\n" + "=" * 60)
print("FINAL OUTPUT")
print("=" * 60)
print(result['final_output'])

---

# 2. Reflexion Agent

**Pattern:** Attempt → Evaluate → Store Learning → Retry (learns from past)

**Goal:** Improve performance over multiple tasks by learning from failures

In [ ]:
from typing import Annotated
from operator import add
from langgraph.checkpoint.memory import MemorySaver

# State for Reflexion Agent
class ReflexionState(TypedDict):
    task: str
    attempt: str
    evaluation: str
    score: int  # 0-10
    learnings: Annotated[list[str], add]  # Accumulate learnings
    attempt_number: int

# Node 1: Generate attempt using past learnings
def generate_attempt(state: ReflexionState):
    """Generate attempt using past experiences"""
    print(f"\n🎯 Attempt #{state['attempt_number']}: Generating solution...")

    # Construct prompt with past learnings
    learnings_text = "\n".join(state['learnings']) if state['learnings'] else "No past learnings yet."

    response = llm.invoke([
        HumanMessage(content=f"""
        Task: {state['task']}

        Past learnings from previous attempts:
        {learnings_text}

        Based on these learnings, provide your best solution.
        """)
    ])

    attempt = response.content
    print(f"Attempt: {attempt[:100]}...")
    return {"attempt": attempt}

# Node 2: Evaluate attempt (simulated external evaluation)
def evaluate_attempt(state: ReflexionState):
    """External evaluation of attempt"""
    print(f"\n📊 Evaluating attempt #{state['attempt_number']}...")

    response = llm.invoke([
        HumanMessage(content=f"""
        Evaluate this solution on a scale of 0-10:

        Task: {state['task']}
        Solution: {state['attempt']}

        Provide:
        1. Score (0-10)
        2. What went wrong (if score < 8)
        3. Specific learning for next attempt

        Format:
        SCORE: X
        EVALUATION: ...
        LEARNING: ...
        """)
    ])

    evaluation = response.content

    # Extract score (simple parsing)
    try:
        score_line = [line for line in evaluation.split('\n') if 'SCORE:' in line][0]
        score = int(score_line.split(':')[1].strip())
    except:
        score = 5  # Default

    # Extract learning
    try:
        learning_line = [line for line in evaluation.split('\n') if 'LEARNING:' in line][0]
        learning = learning_line.split(':', 1)[1].strip()
    except:
        learning = "Try a different approach"

    print(f"Score: {score}/10")
    print(f"Learning: {learning}")

    return {
        "evaluation": evaluation,
        "score": score,
        "learnings": [f"Attempt {state['attempt_number']}: {learning}"]
    }

# Node 3: Check if retry needed
def should_retry(state: ReflexionState) -> Literal["retry", "finish"]:
    """Decide if we need to retry"""
    if state['score'] >= 8:
        print(f"\n✅ Score {state['score']}/10 is good! Finishing...")
        return "finish"

    if state['attempt_number'] >= 3:
        print(f"\n⚠️ Max attempts reached. Finishing...")
        return "finish"

    print(f"\n🔄 Score {state['score']}/10 is low. Retrying with learnings...")
    return "retry"

# Node 4: Increment attempt
def increment_attempt(state: ReflexionState):
    """Increment attempt counter"""
    return {"attempt_number": state['attempt_number'] + 1}

# Build Reflexion Agent Graph
reflexion_workflow = StateGraph(ReflexionState)

# Add nodes
reflexion_workflow.add_node("attempt", generate_attempt)
reflexion_workflow.add_node("evaluate", evaluate_attempt)
reflexion_workflow.add_node("increment", increment_attempt)

# Add edges
reflexion_workflow.add_edge(START, "attempt")
reflexion_workflow.add_edge("attempt", "evaluate")
reflexion_workflow.add_conditional_edges(
    "evaluate",
    should_retry,
    {
        "retry": "increment",
        "finish": END
    }
)
reflexion_workflow.add_edge("increment", "attempt")  # Loop back

# Compile with checkpointer (stores learnings)
reflexion_agent = reflexion_workflow.compile(checkpointer=MemorySaver())

## Test Reflexion Agent (Single Task)

In [ ]:
# Run Reflexion Agent on a single task
print("=" * 60)
print("REFLEXION AGENT: Learning from Evaluation")
print("=" * 60)

config = {"configurable": {"thread_id": "task-1"}}

result = reflexion_agent.invoke({
    "task": "Write a function to find the longest palindrome substring",
    "attempt": "",
    "evaluation": "",
    "score": 0,
    "learnings": [],
    "attempt_number": 1
}, config)

print("\n" + "=" * 60)
print("FINAL RESULT")
print("=" * 60)
print(f"Final Score: {result['score']}/10")
print(f"Total Attempts: {result['attempt_number']}")
print(f"\nFinal Solution:\n{result['attempt']}")
print(f"\nAll Learnings:")
for learning in result['learnings']:
    print(f"  - {learning}")

## Test Reflexion Agent (Multiple Tasks - Shows Learning Across Tasks)

In [ ]:
# Run Reflexion Agent on multiple similar tasks
# Shows how learnings accumulate and improve performance

tasks = [
    "Write a function to reverse a string",
    "Write a function to check if a string is a palindrome",
    "Write a function to find all anagrams in a list of strings"
]

print("=" * 60)
print("REFLEXION AGENT: Learning Across Multiple Tasks")
print("=" * 60)

# Use same thread_id to accumulate learnings across tasks
config = {"configurable": {"thread_id": "string-tasks"}}

for i, task in enumerate(tasks, 1):
    print(f"\n{'='*60}")
    print(f"TASK {i}: {task}")
    print("=" * 60)

    result = reflexion_agent.invoke({
        "task": task,
        "attempt": "",
        "evaluation": "",
        "score": 0,
        "learnings": [],  # Will accumulate from previous tasks
        "attempt_number": 1
    }, config)

    print(f"\n✅ Task {i} completed with score: {result['score']}/10")
    print(f"Total learnings accumulated: {len(result['learnings'])}")

print("\n" + "=" * 60)
print("ALL ACCUMULATED LEARNINGS")
print("=" * 60)
for learning in result['learnings']:
    print(f"  - {learning}")

---

# Key Differences Summary

In [ ]:
## Reflection Agent
# Single task, multiple refinements
task = "Write about AI"

# Iteration 1
draft_1 = generate(task)
critique_1 = critique(draft_1)
draft_2 = refine(draft_1, critique_1)

# Iteration 2
critique_2 = critique(draft_2)
final = refine(draft_2, critique_2)

**Focus:** Improve current output quality

---

In [ ]:
## Reflexion Agent
# Multiple tasks, learns from failures
learnings = []

# Task 1
attempt_1 = generate(task_1, learnings)  # No learnings yet
score_1, learning_1 = evaluate(attempt_1)  # Score: 3/10
learnings.append(learning_1)

# Task 2
attempt_2 = generate(task_2, learnings)  # Uses learning_1
score_2, learning_2 = evaluate(attempt_2)  # Score: 6/10
learnings.append(learning_2)

# Task 3
attempt_3 = generate(task_3, learnings)  # Uses learning_1 + learning_2
score_3 = evaluate(attempt_3)  # Score: 9/10 (improved!)

**Focus:** Learn from past failures to improve future attempts

---

## When to Use

**Use Reflection Agent when:**

- You want to improve a single output

- Quality matters more than speed

- Task is one-time (essay, report, code)

**Use Reflexion Agent when:**

- You have multiple similar tasks

- Agent can learn from failures

- External evaluation is available

- Performance improves over time (games, debugging, optimization)